# Explode
## Explode by one columns
Transform each element of a list-like to a row, replicating index values.

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

df = pd.read_csv("../08-core_data_manipulation/data/IMDB Top 250 Movies.csv")

# let's clean the data type
numeric_columns = ["budget",'box_office']

for col in numeric_columns:
    df[col] = df[col].apply(pd.to_numeric, errors='coerce')

df

As you can see, the `genre` column contain multiple values. We wanted to create a new dataframe with one genre per row
### Explode genres

We notice that the `genre` column is not exactly a list, so we cannot explode it directly.

In fact, if you explode it directly, nothing really happens. See the cell below.

In [ ]:
df['genre'].explode().head()

Therefore, we need to transform this `genre` column to lists or other iterable types.
- We can create a new column called `genre_list` using string operations we have learnt
- explode by `genre_list` instead of `genre` itself.

In [ ]:
df['genre_list'] = df['genre'].str.split(",")
df[['name', 'genre_list']].head()

In [ ]:
# Checking the types and confirm the type of genre_list is list
print(type(df['genre'].iloc[0]))
print(type(df['genre_list'].iloc[0]))

In [ ]:
df_exploded = df.explode('genre_list', ignore_index=True)

df_exploded[['rank', 'name','genre','genre_list']]

Now we can inspect genres

In [ ]:
df_exploded['genre_list'].value_counts()

In [ ]:
# there are 21 genres

df_exploded['genre_list'].nunique()

In [ ]:
# Let's plot the distribution of ratings of the top five genres

sns.kdeplot(df_exploded[df_exploded['genre_list'].isin(df_exploded['genre_list'].value_counts().head(5).index)], x='rating', hue='genre_list',
            fill=True)


In [ ]:
sns.boxenplot(df_exploded[df_exploded['genre_list'].isin(df_exploded['genre_list'].value_counts().head(5).index)], y='rating', hue='genre_list',
            gap=0.2)

### Explode actors (casts)

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

df = pd.read_csv("../08-core_data_manipulation/data/IMDB Top 250 Movies.csv")

# let's clean the data type
numeric_columns = ["budget",'box_office']

for col in numeric_columns:
    df[col] = df[col].apply(pd.to_numeric, errors='coerce')

df

In [ ]:
df['cast_list'] = df['casts'].str.split(",")
df_explode_cast = df.explode('cast_list', ignore_index=True)

df_explode_cast

Now we wanted to see who cast in most top movies.


In [ ]:
df_explode_cast['cast_list'].value_counts()

And who generate the most box office?

In [ ]:
cast_box_office_total = df_explode_cast.groupby('cast_list', as_index=False)['box_office'].sum().sort_values(by='box_office',ascending=False)
cast_box_office_total

### Explode by directors

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

df = pd.read_csv("../08-core_data_manipulation/data/IMDB Top 250 Movies.csv")

# let's clean the data type
numeric_columns = ["budget",'box_office']

for col in numeric_columns:
    df[col] = df[col].apply(pd.to_numeric, errors='coerce')

df

In [ ]:
df['director_list']=df['directors'].str.split(",")
df['director_list']

In [ ]:
df_exploded_directors = df.explode('director_list', ignore_index=True)

In [ ]:
df_exploded_directors['director_list'].value_counts()

In [ ]:
# whose budget is the highest

df_exploded_directors.groupby("director_list")['budget'].sum().reset_index().sort_values(by='budget',ascending=False)

## Explode by multiple columns (sequential)
### Cast and director
Let's say, we wanted to study which cast collaborate with which director the most

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

df = pd.read_csv("../08-core_data_manipulation/data/IMDB Top 250 Movies.csv")

# let's clean the data type
numeric_columns = ["budget",'box_office']

for col in numeric_columns:
    df[col] = df[col].apply(pd.to_numeric, errors='coerce')

df

In [ ]:
df['cast_list'] = df['casts'].str.split(",")
df['director_list']=df['directors'].str.split(",")

df_explode_cast_director = df.explode('cast_list', ignore_index=True).explode('director_list', ignore_index=True)
df_explode_cast_director

In [ ]:
df_explode_cast_director[['cast_list','director_list']].value_counts()

In [ ]:
cast_name, director_name = 'Takashi Shimura', 'Akira Kurosawa'
df_explode_cast_director[(df_explode_cast_director['cast_list']==cast_name) & (df_explode_cast_director['director_list']==director_name)]

In [ ]:
# Another way to do it is to create a list of movies in the grouped dataframe

df_explode_cast_director.groupby(['cast_list','director_list'], as_index=False).agg(
    count=('cast_list', 'count'),
    movies=('name', list),
    box_office_million=('box_office', lambda x: (x.sum()/1_000_000))
).sort_values(by='count',ascending=False)

### Director and writer

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

df = pd.read_csv("../08-core_data_manipulation/data/IMDB Top 250 Movies.csv")

# let's clean the data type
numeric_columns = ["budget",'box_office']

for col in numeric_columns:
    df[col] = df[col].apply(pd.to_numeric, errors='coerce')

df

In [ ]:
director_write_pair = (df.assign(
    director_list = df['directors'].str.split(","),
    writer_list = df['writers'].str.split(",")
).
explode('director_list', ignore_index=True).explode('writer_list', ignore_index=True).
                       groupby(['director_list','writer_list'], as_index=False).agg(
    count=('director_list', 'count'),
    movies=('name', list),
    box_office_million=('box_office', lambda x: (x.sum()/1_000_000)),
    mean_rating=('rating', 'mean')
)).assign(
    mean_box_office = lambda x: x['box_office_million']/x['count']
).sort_values(by='mean_rating',ascending=False)

director_write_pair